# Transisi Energi Hijau di Indonesia

**Periode:** 17 April – 18 Mei 2026  
**Sumber:** Semantik API (keyword-filtered + entity-level dengan topic_keywords)  
**Kata kunci:** transisi energi, energi hijau, EBT, energi baru terbarukan, energi surya, geothermal, PLTS, PLTA, biomassa, cofiring, CCS, karbon

> Semua data — volume, sumber, sentimen entitas, timeline, ko-okurensi, dan framing — menggunakan topic_keywords filtering.  
> Tanpa filter sebelumnya, data entitas mencakup semua artikel lintas topik. Sekarang semua endpoint entity-level mendukung `topic_keywords` sehingga data benar-benar spesifik pada transisi energi hijau.

In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (10, 5)
plt.style.use('seaborn-v0_8-whitegrid')

DATA_DIR = 'data'

def load_json(filename):
    with open(os.path.join(DATA_DIR, filename)) as f:
        return json.load(f)

source_data = load_json('source_comparison.json')
trend_data = load_json('topic_trend.json')
articles = load_json('articles_search.json')

total_articles = sum(s['article_count'] for s in source_data)
print(f'Total artikel: {total_articles} dari {len(source_data)} media')
print(f'Artikel sample: {len(articles)}')
print(f'Periode: {trend_data[0]["date"]} — {trend_data[-1]["date"]}')

## 1. Volume & Sumber Artikel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sources = [s['source'] for s in source_data]
counts = [s['article_count'] for s in source_data]
colors = plt.cm.viridis([i/len(sources) for i in range(len(sources))])

axes[0].barh(sources, counts, color=colors)
axes[0].set_xlabel('Jumlah Artikel')
axes[0].set_title('Artikel per Media')
axes[0].invert_yaxis()

pos = [s['sentiment_distribution']['positive'] for s in source_data]
neg = [s['sentiment_distribution']['negative'] for s in source_data]
neu = [s['sentiment_distribution']['neutral'] for s in source_data]

axes[1].barh(sources, pos, label='Positif', color='#2ecc71')
axes[1].barh(sources, neg, left=pos, label='Negatif', color='#e74c3c')
axes[1].barh(sources, neu, left=[p+n for p,n in zip(pos,neg)], label='Netral', color='#95a5a6')
axes[1].set_xlabel('Jumlah Artikel')
axes[1].set_title('Distribusi Sentimen per Media')
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

### Volume Mingguan

In [ ]:
from datetime import datetime
from collections import defaultdict

weekly = defaultdict(int)
for d in trend_data:
    dt = datetime.strptime(d['date'], '%Y-%m-%d')
    week = dt.isocalendar()[1]
    weekly[week] += d['article_count']

weeks = sorted(weekly.keys())

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f'W{w}' for w in weeks], [weekly[w] for w in weeks], color='#3498db')
for bar, val in zip(bars, [weekly[w] for w in weeks]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), ha='center', fontweight='bold')
ax.set_ylabel('Jumlah Artikel')
ax.set_title('Volume Artikel per Minggu (Keyword-Filtered)')
plt.tight_layout()
plt.show()

## 2. Sentimen Entitas (Topic-Filtered ✅)

Data sentimen sekarang menggunakan `topic_keywords` pada entity-level endpoints. Angka mencerminkan hanya artikel dalam topik transisi energi hijau, bukan semua artikel yang menyebut entitas tersebut.

Perbandingan dengan laporan sebelumnya (tanpa filter topik):

In [ ]:
# Old data (previous report, unscoped)
old_data = {
    'pertamina': {'articles': 307, 'score': 2.76},
    'pln': {'articles': 113, 'score': -1.03},
    'prabowo': {'articles': 2109, 'score': 1.45},
    'bahlil': {'articles': 231, 'score': 2.14},
    'nikel': {'articles': 49, 'score': 1.06},
    'plts':  {'articles': 39,  'score': 1.79},
    'ebt':   {'articles': 2,   'score': 16.0},
    'karbon':{'articles': 3,   'score': 1.67}
}

entities_to_show = ['pertamina', 'pln', 'prabowo', 'bahlil', 'nikel', 'plts', 'ebt', 'karbon']

print(f"{'Entitas':<12} {'Lama':>8} {'Baru':>5} {'Δ':>5} | {'Sentimen Lama':>14} {'Baru':>14}")
print("-" * 65)

new_sentiments = {}
for e in entities_to_show:
    try:
        s = load_json(f'{e}_sentiment.json')
        new_sentiments[e] = s
        old = old_data.get(e, {})
        old_art = old.get('articles', 0)
        new_art = s.get('article_count', 0)
        delta = new_art - old_art
        old_sc = old.get('score', 0)
        new_sc = s.get('average_score', 0)
        print(f"{e:<12} {old_art:>8} {new_art:>5} {delta:+5d} | {old_sc:>+8.2f}     {new_sc:>+6.2f}")
    except FileNotFoundError:
        print(f"{e:<12} No data")
    except Exception as ex:
        print(f"{e:<12} Error: {ex}")

In [ ]:
# Sentiment pie charts — only entities with topic-scoped data
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
plot_entities = ['pertamina', 'pln', 'prabowo', 'plts', 'nikel', 'bahlil']

for i, e in enumerate(plot_entities):
    ax = axes[i//3][i%3]
    data = new_sentiments.get(e, {})
    if not data:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        continue
    labels = ['Positif', 'Negatif', 'Netral']
    values = [data.get('positive', 0), data.get('negative', 0), data.get('neutral', 0)]
    colors = ['#2ecc71', '#e74c3c', '#95a5a6']
    art_count = data.get('article_count', 0)
    avg_sc = data.get('average_score', 0)
    ax.pie(values, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
    ax.set_title(f'{e.upper()}\nSkor: {avg_sc:+.2f} | {art_count} artikel (topic-filtered)')

plt.suptitle('Distribusi Sentimen per Entitas (Topic-Filtered ✅)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. Timeline Sebutan Harian (Topic-Filtered ✅)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True)
plot_entities = ['plts', 'pln', 'pertamina', 'prabowo', 'karbon', 'ebt']

for i, e in enumerate(plot_entities):
    ax = axes[i//3][i%3]
    timeline = load_json(f'{e}_timeline.json')
    if not timeline:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        ax.set_title(e.upper())
        continue
    dates = [d['date'] for d in timeline]
    mentions = [d['mention_count'] for d in timeline]
    color = ['#f39c12', '#3498db', '#9b59b6', '#e74c3c', '#1abc9c', '#e67e22'][i]
    ax.fill_between(range(len(dates)), mentions, alpha=0.2, color=color)
    ax.plot(range(len(dates)), mentions, color=color, linewidth=1.5)
    ax.set_ylabel('Sebutan/hari')
    ax.set_title(f'{e.upper()} (topic-filtered)')
    if mentions:
        max_idx = mentions.index(max(mentions))
        ax.annotate(f'{dates[max_idx]}: {mentions[max_idx]}', xy=(max_idx, mentions[max_idx]),
                    xytext=(min(max_idx+3, len(dates)), max(mentions)*0.9),
                    arrowprops=dict(arrowstyle='->', color='gray'), fontsize=8)

plt.suptitle('Sebutan Harian per Entitas (Topic-Filtered ✅)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Ko-okurensi Entitas (Topic-Filtered ✅)

Entitas mana yang sering muncul bersamaan dalam topik transisi energi hijau.

In [ ]:
cooc_entities = ['plts', 'pln', 'pertamina', 'prabowo']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, entity in enumerate(cooc_entities):
    ax = axes[idx//2][idx%2]
    data = load_json(f'{entity}_cooccurrence.json')
    coocs = data.get('co_occurring_entities', [])[:8]
    if not coocs:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        ax.set_title(f'{entity.upper()}')
        continue
    labels = [c['word'] for c in coocs]
    values = [c['co_occurrence_count'] for c in coocs]
    colors = plt.cm.RdYlGn([v/max(values) if values else 0 for v in values])
    ax.barh(labels, values, color=colors)
    ax.set_title(f'{entity.upper()} ({data.get("mention_count", 0)} sebutan, topic-filtered)')
    ax.invert_yaxis()
    for j, v in enumerate(values):
        ax.text(v + 0.3, j, str(v), va='center', fontsize=9)

plt.suptitle('Ko-okurensi Entitas (Topic-Filtered ✅)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Framing Analisis (Topic-Filtered ✅)

Bagaimana media membingkai setiap entitas dalam konteks transisi energi hijau.

In [ ]:
framing_entities = ['pln', 'plts', 'pertamina', 'prabowo', 'nikel']

for entity in framing_entities:
    framing = load_json(f'{entity}_framing.json')
    print(f"\n{'='*60}")
    print(f"FRAMING: {entity.upper()} (topic-filtered)")
    print('='*60)
    if isinstance(framing, dict) and 'by_source' in framing:
        for source, frames in list(framing['by_source'].items())[:4]:
            print(f"\n📰 {source}:")
            for frame in frames[:3]:
                phrase = frame.get('framing_phrase', 'N/A')
                count = frame.get('article_count', 0)
                print(f"  • {phrase} ({count} artikel)")
    elif isinstance(framing, list):
        for frame in framing[:8]:
            print(f"  • {frame['framing_phrase']} ({frame['article_count']} artikel)")

## 6. Temuan Kunci

1. **PLTS adalah entitas energi hijau paling dominan** — 39 sebutan dalam topik, sentimen +1.79. Satu-satunya entitas yang sepenuhnya relevan dengan transisi energi. Framing mencakup PLTS atap, PLTS Mentari Nusantara I (1.225 MW), target 100 GW Prabowo, dan ekspansi ke Bangladesh.

2. **Pertamina jauh lebih kecil dengan topic filter** — dari 307 menjadi 9 artikel. Framing Pertamina dalam konteks hijau adalah bioetanol Lampung, minyak jelantah, dan LanzaTech — bukan BBM.

3. **PLN berbalik sentimen: dari −1.03 menjadi +2.31** — tanpa filter, data PLN tercampur artikel pemadaman listrik Jakarta. Dengan filter topik, framing PLN adalah cofiring biomassa, PLTS, smart and green building, dan ekspansi Bangladesh.

4. **Prabowo 12 sebutan dalam topik** (dari 2.109 tanpa filter) — framing terkait PLTS 100 GW, target energi hijau, dan deregulasi ESDM.

5. **EBT (2 artikel) dan Karbon (3 artikel) masih sangat minim** — topik carbon trading dan bursa karbon belum masuk pemberitaan mainstream. Ko-okurensi karbon dengan Bank Mandiri dan IDX menunjukkan fokus ke bursa karbon.

6. **Nikel masih minim dalam konteks hijau** — hanya 1 artikel dengan topic filter. Meskipun nikel penting untuk baterai EV, pemberitaan nikel lebih banyak terkait ekstraksi dan batu bara.

### Perubahan Metodologi

- ✅ **Semua endpoint entity-level sekarang mendukung `topic_keywords`** — data entitas (sentimen, timeline, ko-okurensi, framing) benar-benar spesifik pada topik transisi energi hijau.
- ❌ Laporan sebelumnya menggunakan entity-level **tanpa filter topik**, sehingga data entitas tercampur dengan artikel di luar topik.
- Dampak: angka sebenarnya 5-20× lebih kecil dari yang dilaporkan sebelumnya. Ini bukan penurunan volume, melainkan koreksi akurasi.

### Keterbatasan
- `articles/search` dibatasi 50 hasil.
- Keyword matching masih menghasilkan noise (entity "debt collector", "polisi" muncul karena artikel mengandung kata-kata yang cocok secara kebetulan).
- `framing/{word}/by-source` untuk beberapa entitas masih berpotensi corrupt JSON.

---
*Data: Semantik API (keyword-filtered + entity-level topic_keywords). 17 Apr – 18 Mei 2026.*